# Cross-product against relatedness, with fits — reference style — 1kg_eur

The curve from `11_crossproduct_curves.ipynb`, redrawn to match the reference
figures in `publication/figures/with_fits/` (e.g. *Standing height with fits*):
one figure per phenotype, bin means as dark points with no error bars, each
regression line drawn only over the range it was fitted on, legend to the
right. The type sizes, spines, grid, marker size and line colours were
measured off the reference, not approximated — the 12 pt bold title matches
its width to the pixel.

**The regression lines are ours, not the reference's.** Each takes the colour
of the reference model it is closest to, and is labelled by model name alone:
a line is the whole fit, not any one coefficient read off it.

| reference | legend | reports | fitted on | design |
|---|---|---|---|---|
| $\hat h^2_{\mathrm{unrel*}}$ | Unrel | `h2_Unrel` | $a_{ij} < 0.02$ | slope through the origin |
| — | OneSlope | `h2_OneSlope` | $a_{ij} < 0.7$ | slope through the origin |
| $\hat h^2_{\mathrm{ped*}}$ | Rel | `h2_Rel` | 0.02–0.7 | slope + intercept |
| $\hat h^2_{\mathrm{ped*f}}$ | OneSlopeOffsets | `h2_OneSlopeOffsets` | 0.09–0.7 | one slope, an intercept per band |
| $\hat h^2_{\mathrm{HS*}}$ (colour only) | RelOffsets | `h2_RelOffsets` | 0.09–0.7 | a slope and an intercept per band |
| $\hat h^2_{\mathrm{FS*}}$ | FS | `b2_FS`, `b2_FS.slope` | 0.4–0.6 | slope + intercept |

OneSlope has no counterpart in the reference and gets a sixth colour.

**Points** are `noPO` — `other` + `FS`, parent-offspring removed — the class
set every model is fitted on. **Parent-offspring pairs are triangles**, their
own series: they sit in the same bins as full sibs but share neither dominance
nor a rearing environment, so they are in no fit and never averaged into the
circles. Bins with fewer than `MIN_PAIRS_PER_BIN` pairs are not drawn, for
either series.

**Lines** are fitted on every populated bin, exactly as `src/he_estimators.py`
fits them — the display cut does not touch them. Every figure checks each
line's coefficients against what `HE.estimates` reports, so this notebook
cannot draw a model other than the one in `estimators_<run>.tsv`.

Where this departs from the reference, deliberately:

- **The y axis is set by the bin means alone**, with matplotlib's 5% margin.
  In the reference the lines count too; here a band line fitted on a few
  sparse bins could otherwise set the axis and flatten the data.
- **The palette is the reference's own, and it fails the colourblind checks**
  (the two reds are ΔE 5.4 apart even with normal vision). It is kept because
  matching was the point; the swap is one dict in the style cell.

**Prerequisites:** `08_accumulate.ipynb` — `*_merged.full.tsv` and
`*_merged.jk.tsv`, plus the bins file they were merged with.

## Setup

In [ ]:
import glob
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

REPO_DIR = os.environ.get("AOUCOV_REPO") or next(
    (p for p in (os.path.expanduser("~/repos/aou-covariance"),
                 os.path.expanduser("~/repos/AOU-covariance"),
                 os.path.expanduser("~/aou-covariance"))
     if os.path.isdir(os.path.join(p, "src"))),
    None,
)
assert REPO_DIR, "cannot find the repo checkout — set AOUCOV_REPO"

WORK_DIR  = os.path.expanduser(f"~/grm_pheno_cov_{RUN_DIR}")
BINS_FILE = f"{WORK_DIR}/bins_wide.txt"      # the bins the merge used

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
OUT_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}/07_estimators/curves_with_fits"
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, f"{REPO_DIR}/src")
import importlib
import he_estimators as HE
importlib.reload(HE)

assert os.path.isfile(BINS_FILE), f"missing {BINS_FILE} — see 08_accumulate.ipynb"
MID = HE.load_grid(BINS_FILE)

# The block count is a property of the merge, so read it off a merged file
# instead of restating it. Only bin_curve needs it here -- no SE is drawn --
# but load_arrays refuses a count that does not match the file.
_JK = sorted(glob.glob(f"{WORK_DIR}/*_merged.jk.tsv"))
assert _JK, f"no *_merged.jk.tsv in {WORK_DIR} — run 08_accumulate.ipynb first"
NBLOCKS = HE.blocks_in(_JK[0])

# Duplicate/MZ pairs sit at a_ij ~ 1 and no estimator uses them; everything is
# drawn over a_ij <= MAX_A, the top of the range the models are fitted on.
MAX_A = HE.R_HI

# Display only. A bin mean resting on a few pairs is mostly noise and reads as
# structure, so it is not drawn -- but the FITS use every populated bin, as
# he_estimators does, so this changes what is shown and never what is fitted.
MIN_PAIRS_PER_BIN = 50

print(f"bins: {len(MID)} | {NBLOCKS} jackknife blocks | drawing a_ij <= {MAX_A}")
print(f"out : {OUT_DIR}")

## Style

Measured off the reference, which is close to matplotlib's defaults: open
axes (left and bottom spines only), a light grid on both axes, a grey zero
line, 10 pt ticks and legend, 11 pt axis labels, a 12 pt bold title over a
9 pt grey subtitle, both flush with the left spine. It is **not** covplot's
style — that is theme_bw with a full panel border — so `CP.apply_style` is
deliberately not called; covplot supplies only the axis labels.

In [ ]:
import covplot as CP
importlib.reload(CP)

# A stale covplot on the VM surfaces as an AttributeError deep in a drawing
# cell; check here, by name and by API level, so the message says what to do.
_missing = [_n for _n in ["mirror_plots", "XLABEL", "YLABEL"]
            if not hasattr(CP, _n)]
assert not _missing, (
    f"src/covplot.py is out of date: {', '.join(_missing)} missing.\n"
    f"  git -C {REPO_DIR} pull      then restart the kernel if it persists")
assert getattr(CP, "API", 0) >= 1, (
    f"src/covplot.py is API {getattr(CP, 'API', 0)}, this notebook needs >= 1.\n"
    f"  git -C {REPO_DIR} pull      then restart the kernel if it persists")

# The reference is set in Helvetica (its G has the spur Arial's lacks), and the
# type sizes below come from text widths measured on it at its 144 dpi: title,
# subtitle, axis labels and ticks each match to within a few pixels. So this
# notebook resolves its own stack rather than covplot's -- Helvetica and its
# Linux clones first, then Arial and its metric twins, which set to the same
# widths. A Linux image may have none of them, and DejaVu Sans sets ~10%
# wider; drop any of these as .ttf/.otf into FONT_DIR to use it without
# installing anything system-wide.
from matplotlib import font_manager
REF_FONTS = ["Helvetica", "Nimbus Sans", "Nimbus Sans L", "TeX Gyre Heros",
             "FreeSans", "Arial", "Liberation Sans", "Arimo", "Helvetica Neue"]
FONT_DIR = os.path.expanduser("~/fonts")
for _f in glob.glob(f"{FONT_DIR}/*.[ot]tf"):
    font_manager.fontManager.addfont(_f)
_have = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((f for f in REF_FONTS if f in _have), "DejaVu Sans")
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "savefig.transparent": False,
    "font.family": FONT, "font.size": 10,
    "mathtext.fontset": "custom", "mathtext.rm": FONT,
    "mathtext.it": f"{FONT}:italic", "mathtext.bf": f"{FONT}:bold",
    "text.color": "black",
    "axes.edgecolor": "black", "axes.linewidth": 0.8,
    "axes.labelsize": 11, "axes.labelcolor": "black",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.which": "major", "axes.axisbelow": True,
    "axes.xmargin": 0.05, "axes.ymargin": 0.05,
    "grid.color": "#e5e5e5", "grid.linewidth": 0.8, "grid.alpha": 1.0,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "xtick.color": "black", "ytick.color": "black",
    "xtick.major.size": 3.5, "ytick.major.size": 3.5,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8,
    "xtick.minor.visible": False, "ytick.minor.visible": False,
    "legend.fontsize": 10, "legend.frameon": False,
})

TITLE_SIZE, SUBTITLE_SIZE, SUBTITLE_COL = 12, 9, "#555555"
POINT = dict(marker="o", ls="none", ms=4.5, mew=0, color="#333333")
# Parent-offspring: same ink, since it is data rather than a fit, and told
# apart by shape. A triangle reads smaller than a circle at the same size, so
# it is set larger; the white ring keeps it separable where it overlaps the
# circle in its own bin. Nudged along x by PO_XOFF, as 11_crossproduct_curves
# does, so a triangle cannot sit squarely on top of that circle.
PO_POINT = dict(marker="^", ls="none", ms=6.5, mew=0.6, mec="white",
                color="#333333")
PO_XOFF = 0.004
ZERO_LINE = dict(color="#b0b0b0", lw=0.8)
# The reference's lines are lighter on the panel than in its legend: measured
# across the same line, the panel copy is ~0.7 alpha at ~1.25 pt, the legend
# key opaque at 1.5 pt, in the same colour. So the key is a proxy, not the line.
LINE_LW, LINE_ALPHA, KEY_LW = 1.25, 0.7, 1.5

# Legend order, top to bottom, is the reference's: the narrow first-degree fit
# first, the unrelated slope last. Colours were sampled from the reference's
# legend handles (the same value, to within compression, in both panels).
# Labels are model names, not estimator symbols. A line is the whole fit: the
# FS line is a slope AND an intercept, and b2_FS is twice the intercept alone,
# so labelling the line with it named one term of the model as if it were all.
#
# The palette validator FAILS this set: the two reds are dE 5.4 apart with
# normal vision, well under the floor of 15, and the worst colourblind pair is
# dE 3.4. It is the reference's palette, kept on purpose. In the reference the
# two reds are told apart by shape -- straight line vs steps -- and the same is
# true of h2_Rel and h2_OneSlopeOffsets here. To swap in the validated ladder
# colours instead, take the four ladder rungs from CP.MODEL_COL.
FIT_STYLE = {
    "b2_FS":              dict(color="#BE9B6E", label="FS"),
    "h2_RelOffsets":      dict(color="#759672", label="RelOffsets"),
    "h2_Rel":             dict(color="#CE7B79", label="Rel"),
    "h2_OneSlopeOffsets": dict(color="#B5716D", label="OneSlopeOffsets"),
    "h2_OneSlope":        dict(color="#8E7DAE", label="OneSlope"),
    "h2_Unrel":           dict(color="#709BBE", label="Unrel"),
}

# Geometry, in inches, measured off the reference: a 4.76 x 2.92 in axes, with
# the legend in the strip to its right. Fixed rather than left to a layout
# engine, so every phenotype's figure has an identical axes box.
AX_W, AX_H = 4.76, 2.92
MARGIN_L, MARGIN_B, MARGIN_T, MARGIN_R = 0.88, 0.64, 0.59, 1.36
FIGSIZE = (MARGIN_L + AX_W + MARGIN_R, MARGIN_B + AX_H + MARGIN_T)
AX_RECT = [MARGIN_L / FIGSIZE[0], MARGIN_B / FIGSIZE[1],
           AX_W / FIGSIZE[0], AX_H / FIGSIZE[1]]
DPI = 220
print(f"font in use: {FONT}" + ("" if FONT in REF_FONTS else
      f"  (none of the reference's fonts found -- see FONT_DIR, {FONT_DIR})"))

## Load the merged combos

One figure per phenotype at the reference transform and covariate set, the
same pair `11_crossproduct_curves.ipynb` draws. Titles are spelled out, as in
the reference; a phenotype missing from `TITLE` falls back to its name.

In [ ]:
def load_tag(tag):
    return HE.load_arrays(f"{WORK_DIR}/{tag}_merged.full.tsv",
                          f"{WORK_DIR}/{tag}_merged.jk.tsv", len(MID), NBLOCKS)


tags = [os.path.basename(p)[: -len("_merged.full.tsv")]
        for p in sorted(glob.glob(f"{WORK_DIR}/*_merged.full.tsv"))
        if os.path.isfile(p.replace(".full.tsv", ".jk.tsv"))]
assert tags, f"no merged combo with both .full.tsv and .jk.tsv under {WORK_DIR}"

meta = pd.DataFrame([t.split("__")[:3] for t in tags],
                    columns=["phenotype", "transform", "covset"])
meta["tag"] = tags
PHENOTYPES = sorted(meta["phenotype"].unique())
TRANSFORMS = sorted(meta["transform"].unique())
COVSETS = sorted(meta["covset"].unique())
REF_TF = "invnorm" if "invnorm" in TRANSFORMS else TRANSFORMS[0]
REF_CS = "base_pcs" if "base_pcs" in COVSETS else COVSETS[0]

TITLE = {
    "height": "Standing height",
    "weight": "Body weight",
    "bmi": "Body mass index (BMI)",
    "waist_circumference": "Waist circumference",
    "hip_circumference": "Hip circumference",
    "waist_hip_ratio": "Waist-to-hip ratio",
    "systolic_bp": "Systolic blood pressure",
    "diastolic_bp": "Diastolic blood pressure",
    "heart_rate": "Heart rate",
    "glucose": "Glucose",
    "hba1c": "Glycated hemoglobin (HbA1c)",
    "hdl_cholesterol": "HDL cholesterol",
    "ldl_cholesterol": "LDL cholesterol",
    "total_cholesterol": "Total cholesterol",
    "triglycerides": "Triglycerides",
    "creatinine": "Creatinine",
    "alt": "Alanine aminotransferase (ALT)",
    "hemoglobin": "Hemoglobin",
    "hematocrit": "Hematocrit",
    "rbc_count": "Red blood cell count",
    "wbc_count": "White blood cell count",
    "platelet_count": "Platelet count",
    "mch": "Mean corpuscular hemoglobin (MCH)",
    "mchc": "Mean corpuscular hemoglobin concentration (MCHC)",
    "mcv": "Mean corpuscular volume (MCV)",
    "neutrophil_count": "Neutrophil count",
    "lymphocyte_count": "Lymphocyte count",
    "monocyte_count": "Monocyte count",
    "basophil_count": "Basophil count",
    "eosinophil_count": "Eosinophil count",
    "alcohol_audit_c_score": "Alcohol use (AUDIT-C score)",
    "cigarettes_per_day": "Cigarettes per day",
    "pss_score": "Perceived stress (PSS score)",
    "social_support_score": "Social support score",
    "loneliness_score": "Loneliness score",
    "eds_score": "Everyday discrimination (EDS score)",
}


def title_of(phe):
    return TITLE.get(phe, phe.replace("_", " ").capitalize())


untitled = [p for p in PHENOTYPES if p not in TITLE]
print(f"{len(tags)} combos | {len(PHENOTYPES)} phenotypes | "
      f"drawing {REF_TF} / {REF_CS}")
if untitled:
    print(f"no title for {', '.join(untitled)} -- using the name")

## Fits

The four ladder rungs are `HE.MODELS` verbatim; `h2_Unrel` and the FS-range
fit behind `b2_FS` are the two estimators outside the ladder that the figure
also draws, spelled the way `HE.estimates` fits them. `check_against_estimates`
reads each line's reported coefficient back out of `HE.estimates` on the full
data and asserts they agree, so a design that drifts here from the module's
fails loudly instead of quietly drawing a model nobody reported.

In [ ]:
_origin = lambda a: a[:, None]
_slope_int = lambda a: np.c_[a, np.ones_like(a)]
FITS = {
    "h2_Unrel": ([(-np.inf, HE.U_HI)], _origin),
    **HE.MODELS,
    "b2_FS": ([HE.FS_RANGE], _slope_int),
}
assert set(FIT_STYLE) == set(FITS), (
    f"FIT_STYLE and FITS disagree: {sorted(set(FIT_STYLE) ^ set(FITS))}")

_NB = len(HE.OFFSET_BANDS)
# what HE.estimates reports for each line, as a function of its coefficients
READOUT = {
    "h2_Unrel":           ("h2_Unrel",           lambda b: b[0]),
    "h2_OneSlope":        ("h2_OneSlope",        lambda b: b[0]),
    "h2_Rel":             ("h2_Rel",             lambda b: b[0]),
    "h2_OneSlopeOffsets": ("h2_OneSlopeOffsets", lambda b: b[0]),
    "h2_RelOffsets":      ("h2_RelOffsets",      lambda b: b[_NB - 1]),  # top band
    "b2_FS":              ("b2_FS",              lambda b: 2 * b[1]),
}
assert set(READOUT) == set(FITS)


def fit_models(S, N):
    """{name: (beta, a_fitted)} on noPO, over every populated bin in range --
    the bins HE.estimates fits, with no MIN_PAIRS_PER_BIN cut."""
    a, m, n, _ = HE.bin_curve(S, N, HE.NOPO, MID, NBLOCKS)
    out = {}
    for name, (ranges, design) in FITS.items():
        inside = np.zeros(len(a), dtype=bool)
        for lo, hi in ranges:
            inside |= (a >= lo) & (a < hi)
        beta = (HE._wls(design(a[inside]), m[inside], n[inside])
                if inside.any() else None)
        out[name] = (beta, a[inside])
    return out


def check_against_estimates(S, N, fits, tag):
    """Assert every line's reported coefficient equals HE.estimates'."""
    est = HE.estimates(S, N, 0, MID)
    for name, (key, readout) in READOUT.items():
        beta = fits[name][0]
        ours = readout(beta) if beta is not None else np.nan
        theirs = est[(key, "noPO")]
        if np.isnan(ours) and np.isnan(theirs):
            continue
        assert np.isclose(ours, theirs, rtol=1e-9, atol=1e-12), (
            f"{tag}: the {name} line gives {key} = {ours:.6g}, "
            f"HE.estimates reports {theirs:.6g}")


print(f"{len(FITS)} lines: {', '.join(FITS)}")

## Drawing

Each line runs over the bins it was fitted on, clipped to the points on the
panel, and never across the panel as an extrapolation. The offset models are
step functions at the band edges, so every line is broken there: one polyline
across an edge would draw a near-vertical spike joining two levels, a segment
that belongs to no fit.

In [ ]:
from matplotlib.lines import Line2D

BAND_EDGES = sorted({e for r in HE.OFFSET_BANDS.values() for e in r})


def segments(lo, hi):
    """[lo, hi] split at every band edge inside it."""
    cuts = [lo] + [e for e in BAND_EDGES if lo < e < hi] + [hi]
    # stop just short of each cut: the band indicator is [lo, hi), so landing
    # exactly on an edge would evaluate the NEXT band's level
    return [(k0, k1 - 1e-9 if k1 < hi else k1)
            for k0, k1 in zip(cuts[:-1], cuts[1:]) if k1 - k0 > 1e-9]


def draw_with_fits(ax, S, N, fits):
    """Bin means, parent-offspring bin means, and every fitted line.

    Returns (legend handles, the number of noPO bins not drawn for having fewer
    than MIN_PAIRS_PER_BIN pairs, whether PO pairs exist but no PO bin cleared
    that cut -- so a missing series is reported, not silently absent)."""
    a, m, n, _ = HE.bin_curve(S, N, HE.NOPO, MID, NBLOCKS)
    window = a <= MAX_A
    keep = window & (n >= MIN_PAIRS_PER_BIN)
    n_drop = int((window & ~keep).sum())
    a, m = a[keep], m[keep]

    ax.axhline(0, zorder=1, **ZERO_LINE)
    handles = ax.plot(a, m, label="Binned data", zorder=2, **POINT)

    ap, mp, npo, _ = HE.bin_curve(S, N, ["PO"], MID, NBLOCKS)
    kp = (ap <= MAX_A) & (npo >= MIN_PAIRS_PER_BIN)
    po_hidden = bool(len(ap)) and not kp.any()
    if kp.any():
        # above the circles, below the fits
        handles += ax.plot(ap[kp] + PO_XOFF, mp[kp], label="Parent–offspring",
                           zorder=2.5, **PO_POINT)

    for name, style in FIT_STYLE.items():
        beta, a_fit = fits[name]
        if beta is None or not len(a) or not len(a_fit):
            continue
        lo, hi = max(a_fit.min(), a.min()), min(a_fit.max(), a.max())
        if hi <= lo:
            continue
        design = FITS[name][1]
        for k0, k1 in segments(lo, hi):
            xs = np.linspace(k0, k1, 60)
            # scaley=False: the axis is the bin means' alone (see the header)
            ax.plot(xs, design(xs) @ beta, color=style["color"], lw=LINE_LW,
                    alpha=LINE_ALPHA, zorder=3, scalex=False, scaley=False)
        handles.append(Line2D([], [], color=style["color"], lw=KEY_LW,
                              label=style["label"]))

    ax.set_xlabel(CP.XLABEL)
    ax.set_ylabel(CP.YLABEL)
    return handles, n_drop, po_hidden


def figure_with_fits(phe, tag):
    S, N = load_tag(tag)
    fits = fit_models(S, N)
    check_against_estimates(S, N, fits, tag)

    fig = plt.figure(figsize=FIGSIZE)
    ax = fig.add_axes(AX_RECT)
    handles, n_drop, po_hidden = draw_with_fits(ax, S, N, fits)

    # title and subtitle flush with the left spine, baselines 15 and 4 pt
    # above the axes, as in the reference
    ax.annotate(title_of(phe), (0, 1), xycoords="axes fraction",
                xytext=(0, 15), textcoords="offset points", ha="left",
                va="baseline", fontsize=TITLE_SIZE, fontweight="bold")
    ax.annotate("Mean per relatedness bin", (0, 1), xycoords="axes fraction",
                xytext=(0, 4), textcoords="offset points", ha="left",
                va="baseline", fontsize=SUBTITLE_SIZE, color=SUBTITLE_COL)
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.03, 1.0),
              borderaxespad=0.0)
    return fig, n_drop, po_hidden


print("drawing helpers defined")

## Every phenotype

Saved as PNG and PDF, one of each per phenotype. The first two are drawn
inline, as the pair the reference shows; the rest go straight to disk.

In [ ]:
# Set to a shorter list to work on one, e.g. DRAW_PHENOTYPES = ["height"].
DRAW_PHENOTYPES = PHENOTYPES
SHOW_INLINE = 2

saved, dropped, skipped, po_hidden = [], {}, [], []
for phe in DRAW_PHENOTYPES:
    row = meta[(meta["phenotype"] == phe) & (meta["transform"] == REF_TF)
               & (meta["covset"] == REF_CS)]
    if row.empty:
        skipped.append(phe)
        continue
    fig, dropped[phe], _hid = figure_with_fits(phe, row["tag"].iloc[0])
    if _hid:
        po_hidden.append(phe)
    stem = f"{OUT_DIR}/{phe}_{REF_TF}_{REF_CS}_with_fits"
    for ext in ("png", "pdf"):
        fig.savefig(f"{stem}.{ext}", dpi=DPI, bbox_inches="tight")
    saved.append(stem)
    plt.show() if len(saved) <= SHOW_INLINE else plt.close(fig)

print(f"{len(saved)} phenotypes -> {OUT_DIR}  (png + pdf each)")
print(f"every line matched HE.estimates for all {len(saved)}")
if any(dropped.values()):
    print(f"bins not drawn for having < {MIN_PAIRS_PER_BIN} pairs: "
          f"up to {max(dropped.values())} per phenotype (fits still use them)")
if po_hidden:
    print(f"no triangles -- PO pairs present but no bin with >= "
          f"{MIN_PAIRS_PER_BIN}: {', '.join(po_hidden)}")
if skipped:
    print(f"no combo at {REF_TF} / {REF_CS} for: {', '.join(skipped)}")

## Mirror locally

In [ ]:
CP.mirror_plots(OUT_DIR, "with_fits", RUN_DIR)

## Copy notebook to bucket

In [ ]:
_nb = os.path.join(REPO_DIR, "notebooks/11b_crossproduct_curves_with_fits.ipynb")
_gs = ("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/"
       f"{PROJECT_DIR}/{RUN_DIR}/notebooks/11b_crossproduct_curves_with_fits.ipynb")
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")